# Chapter 6: Machine Learning

```{admonition} Learning Objectives
:class: tip
- Understand supervised vs unsupervised learning
- Implement linear regression and gradient descent
- Build classification models: Logistic Regression, SVM, Decision Trees
- Evaluate models with cross-validation
- Handle overfitting with regularization
- Compare different learning algorithms
```

## 6.1 Introduction to Machine Learning

Machine Learning is the **inductive** approach to AI - learning patterns from data rather than encoding explicit rules.

### Paradigm Shift

| Deductive (Chapters 1-5) | Inductive (Chapters 6-10) |
|--------------------------|---------------------------|
| Expert knowledge → Rules | Data → Patterns |
| Explicit programming | Learn from examples |
| Chess evaluation functions | AlphaZero learning |
| Logic rules | Statistical models |

### Types of Learning

1. **Supervised Learning**: Learn $f: X \to Y$ from labeled data
2. **Unsupervised Learning**: Discover structure in unlabeled data
3. **Reinforcement Learning**: Learn from rewards/penalties

In [ ]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, mean_squared_error, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
np.random.seed(42)

print("✓ Libraries imported!")

## 6.2 Linear Regression

The simplest supervised learning model: predict continuous outputs.

### Model

$$\hat{y} = w_0 + w_1 x_1 + \cdots + w_d x_d = \mathbf{w}^T \mathbf{x}$$

### Loss Function (Mean Squared Error)

$$L(\mathbf{w}) = \frac{1}{n} \sum_{i=1}^n (y_i - \mathbf{w}^T \mathbf{x}_i)^2$$

### Closed-Form Solution

$$\mathbf{w}^* = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}$$

In [ ]:
class LinearRegression:
    """Linear Regression with closed-form and gradient descent"""
    
    def __init__(self):
        self.weights = None
    
    def fit_closed_form(self, X, y):
        """Closed-form solution using normal equation"""
        # Add bias term
        X_bias = np.c_[np.ones(len(X)), X]
        
        # w = (X^T X)^-1 X^T y
        self.weights = np.linalg.inv(X_bias.T @ X_bias) @ X_bias.T @ y
        return self
    
    def fit_gradient_descent(self, X, y, learning_rate=0.01, epochs=1000, verbose=False):
        """Gradient descent optimization"""
        X_bias = np.c_[np.ones(len(X)), X]
        n_samples, n_features = X_bias.shape
        
        # Initialize weights
        self.weights = np.zeros(n_features)
        losses = []
        
        for epoch in range(epochs):
            # Predictions
            y_pred = X_bias @ self.weights
            
            # Loss
            loss = np.mean((y - y_pred) ** 2)
            losses.append(loss)
            
            # Gradient: dL/dw = -2/n * X^T (y - y_pred)
            gradient = -2/n_samples * X_bias.T @ (y - y_pred)
            
            # Update
            self.weights -= learning_rate * gradient
            
            if verbose and epoch % 100 == 0:
                print(f"Epoch {epoch}: Loss = {loss:.4f}")
        
        return self, losses
    
    def predict(self, X):
        """Make predictions"""
        X_bias = np.c_[np.ones(len(X)), X]
        return X_bias @ self.weights
    
    def score(self, X, y):
        """R-squared score"""
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        return 1 - (ss_res / ss_tot)

print("✓ Linear Regression implemented!")

In [ ]:
# Example: Linear Regression
X, y = make_regression(n_samples=100, n_features=1, noise=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Closed-form
model_cf = LinearRegression().fit_closed_form(X_train, y_train)
print(f"Closed-form R²: {model_cf.score(X_test, y_test):.4f}")

# Gradient descent
model_gd, losses = LinearRegression().fit_gradient_descent(X_train, y_train, learning_rate=0.1, epochs=500)
print(f"Gradient Descent R²: {model_gd.score(X_test, y_test):.4f}")

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Regression line
ax1.scatter(X_test, y_test, alpha=0.6, label='Data')
X_line = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
ax1.plot(X_line, model_cf.predict(X_line), 'r-', label='Model', linewidth=2)
ax1.set_xlabel('X')
ax1.set_ylabel('y')
ax1.set_title('Linear Regression Fit')
ax1.legend()
ax1.grid(True)

# Loss curve
ax2.plot(losses)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('MSE Loss')
ax2.set_title('Gradient Descent Convergence')
ax2.grid(True)

plt.tight_layout()
plt.show()

## 6.3 Logistic Regression

Binary classification using the logistic (sigmoid) function.

### Model

$$P(y=1|\mathbf{x}) = \sigma(\mathbf{w}^T \mathbf{x}) = \frac{1}{1 + e^{-\mathbf{w}^T \mathbf{x}}}$$

### Loss Function (Cross-Entropy)

$$L(\mathbf{w}) = -\frac{1}{n} \sum_{i=1}^n [y_i \log(\hat{y}_i) + (1-y_i) \log(1-\hat{y}_i)]$$

### Gradient

$$\nabla L = \frac{1}{n} \mathbf{X}^T (\hat{\mathbf{y}} - \mathbf{y})$$

In [ ]:
class LogisticRegression:
    """Logistic Regression for binary classification"""
    
    def __init__(self, learning_rate=0.01, epochs=1000, reg_lambda=0.0):
        self.lr = learning_rate
        self.epochs = epochs
        self.reg_lambda = reg_lambda
        self.weights = None
    
    def sigmoid(self, z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))
    
    def fit(self, X, y, verbose=False):
        X_bias = np.c_[np.ones(len(X)), X]
        n_samples, n_features = X_bias.shape
        
        self.weights = np.zeros(n_features)
        losses = []
        
        for epoch in range(self.epochs):
            # Forward pass
            z = X_bias @ self.weights
            y_pred = self.sigmoid(z)
            
            # Loss (cross-entropy + L2 regularization)
            loss = -np.mean(y * np.log(y_pred + 1e-15) + (1-y) * np.log(1-y_pred + 1e-15))
            if self.reg_lambda > 0:
                loss += self.reg_lambda * np.sum(self.weights[1:]**2) / (2*n_samples)
            losses.append(loss)
            
            # Gradient
            gradient = X_bias.T @ (y_pred - y) / n_samples
            if self.reg_lambda > 0:
                gradient[1:] += self.reg_lambda * self.weights[1:] / n_samples
            
            # Update
            self.weights -= self.lr * gradient
            
            if verbose and epoch % 100 == 0:
                print(f"Epoch {epoch}: Loss = {loss:.4f}")
        
        return self, losses
    
    def predict_proba(self, X):
        X_bias = np.c_[np.ones(len(X)), X]
        return self.sigmoid(X_bias @ self.weights)
    
    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)
    
    def score(self, X, y):
        return accuracy_score(y, self.predict(X))

print("✓ Logistic Regression implemented!")

In [ ]:
# Example: Binary Classification
X, y = make_classification(n_samples=500, n_features=2, n_informative=2, 
                          n_redundant=0, n_clusters_per_class=1, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
model_lr = LogisticRegression(learning_rate=0.1, epochs=1000, reg_lambda=0.1)
model_lr, losses_lr = model_lr.fit(X_train, y_train, verbose=False)

# Evaluate
train_acc = model_lr.score(X_train, y_train)
test_acc = model_lr.score(X_test, y_test)
print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

# Confusion matrix
y_pred = model_lr.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
print(f"\nConfusion Matrix:\n{cm}")

## 6.4 Support Vector Machines

SVMs find the **maximum margin** hyperplane separating classes.

### Primal Form

Minimize: $\frac{1}{2}\|\mathbf{w}\|^2 + C\sum_{i=1}^n \xi_i$

Subject to: $y_i(\mathbf{w}^T\mathbf{x}_i + b) \geq 1 - \xi_i$, $\xi_i \geq 0$

### Kernel Trick

$$K(\mathbf{x}_i, \mathbf{x}_j) = \phi(\mathbf{x}_i)^T \phi(\mathbf{x}_j)$$

Common kernels:
- **Linear**: $K(\mathbf{x}_i, \mathbf{x}_j) = \mathbf{x}_i^T \mathbf{x}_j$
- **RBF**: $K(\mathbf{x}_i, \mathbf{x}_j) = \exp(-\gamma \|\mathbf{x}_i - \mathbf{x}_j\|^2)$
- **Polynomial**: $K(\mathbf{x}_i, \mathbf{x}_j) = (\mathbf{x}_i^T \mathbf{x}_j + c)^d$

In [ ]:
from sklearn.svm import SVC

# Compare different kernels
X_nl, y_nl = make_classification(n_samples=300, n_features=2, n_informative=2,
                                n_redundant=0, n_clusters_per_class=1,
                                class_sep=0.5, random_state=42)

kernels = ['linear', 'rbf', 'poly']
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, kernel in zip(axes, kernels):
    svm = SVC(kernel=kernel, C=1.0, gamma='auto')
    svm.fit(X_nl, y_nl)
    
    # Plot decision boundary
    xx, yy = np.meshgrid(np.linspace(X_nl[:, 0].min()-1, X_nl[:, 0].max()+1, 200),
                        np.linspace(X_nl[:, 1].min()-1, X_nl[:, 1].max()+1, 200))
    Z = svm.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlBu')
    ax.scatter(X_nl[:, 0], X_nl[:, 1], c=y_nl, cmap='RdYlBu', edgecolors='k')
    ax.set_title(f'SVM with {kernel} kernel\nAccuracy: {svm.score(X_nl, y_nl):.3f}')
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

plt.tight_layout()
plt.show()

print("✓ SVM kernels compared!")

## 6.5 Decision Trees

Tree-based models that recursively split data based on feature values.

### Splitting Criterion

**Gini Impurity**:
$$G = 1 - \sum_{k=1}^K p_k^2$$

**Information Gain** (Entropy):
$$H = -\sum_{k=1}^K p_k \log_2 p_k$$

**Advantages:**
- Interpretable (white box)
- Handles non-linear relationships
- No feature scaling needed

**Disadvantages:**
- Prone to overfitting
- High variance

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

# Train decision tree
dt = DecisionTreeClassifier(max_depth=3, random_state=42)
dt.fit(X_train, y_train)

print(f"Decision Tree Accuracy: {dt.score(X_test, y_test):.4f}")

# Visualize tree
plt.figure(figsize=(16, 8))
plot_tree(dt, filled=True, feature_names=['Feature 1', 'Feature 2'], 
         class_names=['Class 0', 'Class 1'], rounded=True)
plt.title('Decision Tree Structure')
plt.show()

print("✓ Decision Tree visualized!")

## 6.6 Model Evaluation and Validation

### Cross-Validation

**K-Fold CV**: Split data into $k$ folds, train on $k-1$, validate on 1.

$$\text{CV Score} = \frac{1}{k} \sum_{i=1}^k \text{Score}_i$$

### Metrics

**Classification:**
- Accuracy: $(TP + TN) / (TP + TN + FP + FN)$
- Precision: $TP / (TP + FP)$
- Recall: $TP / (TP + FN)$
- F1-Score: $2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$

**Regression:**
- MSE: $\frac{1}{n}\sum (y_i - \hat{y}_i)^2$
- MAE: $\frac{1}{n}\sum |y_i - \hat{y}_i|$
- R²: $1 - \frac{SS_{res}}{SS_{tot}}$

In [ ]:
# Compare models with cross-validation
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB

models = {
    'Logistic Regression': LogisticRegression(learning_rate=0.1, epochs=500),
    'SVM (RBF)': SVC(kernel='rbf'),
    'Decision Tree': DecisionTreeClassifier(max_depth=5),
    'Random Forest': RandomForestClassifier(n_estimators=100),
    'Naive Bayes': GaussianNB()
}

print("Model Comparison (5-Fold CV):")
print("=" * 50)

for name, model in models.items():
    if name == 'Logistic Regression':
        # Our custom implementation
        model.fit(X_train, y_train)
        score = model.score(X_test, y_test)
        print(f"{name:20s}: {score:.4f}")
    else:
        # Sklearn models
        scores = cross_val_score(model, X, y, cv=5)
        print(f"{name:20s}: {scores.mean():.4f} (±{scores.std():.4f})")

print("\n✓ Model comparison complete!")

## 6.7 Regularization

Prevent overfitting by penalizing model complexity.

### L2 Regularization (Ridge)

$$L(\mathbf{w}) = \text{MSE}(\mathbf{w}) + \lambda \|\mathbf{w}\|_2^2$$

### L1 Regularization (Lasso)

$$L(\mathbf{w}) = \text{MSE}(\mathbf{w}) + \lambda \|\mathbf{w}\|_1$$

**L1 vs L2:**
- L1: **Sparse** solutions (feature selection)
- L2: **Small** weights (prevents large coefficients)

## 6.8 Summary

### Algorithm Comparison

| Model | Type | Pros | Cons |
|-------|------|------|------|
| Linear Regression | Regression | Simple, interpretable | Linear only |
| Logistic Regression | Classification | Probabilistic, fast | Linear boundary |
| SVM | Classification | Kernel trick, robust | Slow on large data |
| Decision Trees | Both | Interpretable, non-linear | Overfits easily |
| Random Forest | Both | Accurate, robust | Black box |

### Key Takeaways

1. **Inductive learning** discovers patterns from data
2. **Gradient descent** is universal optimization technique
3. **Regularization** prevents overfitting
4. **Cross-validation** gives reliable performance estimates
5. **No free lunch** - no single best algorithm!

## Exercises

```{exercise} Polynomial Regression
:label: ex-6-1

Extend LinearRegression to handle polynomial features. Compare degrees 1, 2, 3, 5 on non-linear data.
```

```{exercise} Multi-class Classification
:label: ex-6-2

Implement One-vs-Rest strategy to extend LogisticRegression to multi-class problems. Test on Iris dataset.
```

```{exercise} Feature Engineering
:label: ex-6-3

Create interaction features $(x_i \cdot x_j)$ and compare model performance with/without them.
```

```{exercise} Bias-Variance Tradeoff
:label: ex-6-4

Plot train vs test error for Decision Trees with varying max_depth. Identify the sweet spot.
```

## Further Reading

- Aggarwal, C. C. (2021). *Artificial Intelligence*, Chapter 6
- Bishop, C. (2006). *Pattern Recognition and Machine Learning*
- Murphy, K. P. (2022). *Probabilistic Machine Learning*
- Hastie et al. (2009). *The Elements of Statistical Learning*

---

**Next Chapter**: [Neural Networks](ch07_neural.ipynb)